Reference implementation for the paper "Automatic Generation of Study Texts from Lecture Slides"

In [4]:
import getpass
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("OpenRouter API key: ")


In [ ]:
from pipeline import OpenRouterProvider

MODEL = "qwen/qwen3.7-plus"
CRITIC = "google/gemini-2.5-flash"

provider = OpenRouterProvider(
    {"professor": MODEL, "researcher": MODEL, "writer": MODEL, "default": MODEL, "critic": CRITIC}
)

In [8]:
# Smoke-test run
# max_slides caps the run so a first pass costs cents; drop it for the whole deck.

from IPython.display import Markdown

from pipeline import run_deck
from runs import lectures

deck = lectures()[0]


def show(src, meta):
    print(
        f"slides {meta['first']:>3}-{meta['last']:<3} {meta['status'] or 'ERROR':9} "
        f"attempts={meta['attempts']} terms={len(meta['search_terms'])} {meta['time_s']}s"
        + (f"  !! {meta['error']}" if meta["error"] else "")
    )


md = await run_deck(provider, deck, window_size=5, max_slides=10, on_window=show)
Markdown(md)

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error: No common ancestor in structure tree

MuPDF error: format error

# Introduction and Optimization Problems

This opening lecture of MIT 6.0002 frames the course as learning to apply computation to real problems through *computational models*, and begins the technical arc with optimization problems, using the knapsack problem as the canonical example.

## Course Mechanics: Prerequisites, Workload, and How 6.0002 Differs from 6.0001

**Instructor and prerequisites.** The course is taught by John Guttag of MIT's Department of Electrical Engineering and Computer Science — a professor and former head of EECS (at roughly 2000 students and 125 faculty, the largest department at MIT), whose research with CSAIL's Networks and Mobile Systems Group spans computer networks and medical applications of AI, and who was inducted as an ACM Fellow in 2006. Three prerequisites are assumed: (1) experience writing object-oriented programs in Python, preferably Python 3.5 (the version used in the course); (2) familiarity with computational complexity — asymptotic notation and reasoning about how running time grows; and (3) familiarity with some simple algorithms. Completing 6.0001 covers everything needed.

**Workload.** Four components:
- **Problem sets** — programming problems designed to do two things: improve your programming skills *and* help you learn the conceptual material. They are not just about getting code to run.
- **Finger exercises** — very small programming problems, each designed to teach a single programming concept; quick workouts rather than big projects.
- **Readings** — from Guttag's textbook *Introduction to Computation and Programming Using Python, with Application to Understanding Data* (2nd edition); seeing the same idea explained a second way often makes it click.
- **Exam** — based on all of the above, so keep up with all three.

The lecture also uses embedded questions (e.g., "Question 1") throughout to keep students thinking.

**Differences from 6.0001.** The programming assignments are somewhat easier — the focus shifts from the programming itself to the problem being solved. The lecture content is more abstract and faster paced. Broadly, the subject is less about learning to program and more about dipping a toe into **data science**: taking the programming foundation you have and using it to understand and extract meaning from data. Programming skills are still honed along the way — you will pick up additional bits of Python, learn software engineering (how to structure programs so they are robust and maintainable), and learn to use packages (leveraging libraries others have written rather than reinventing everything). As for how you improve: "How do you get to Carnegie Hall? Practice, practice, practice" — and the problem sets and finger exercises are where that practice happens.

## The Complexity Toolkit You Are Expected to Bring

Complexity is a stated prerequisite, and it explains both how we compare algorithms and why the knapsack problem is interesting and hard.

- **Definition.** The computational complexity of an algorithm is the amount of resources required to run it — chiefly computation time (measured by the number of elementary operations) and memory. The complexity of a *problem* is the complexity of the best algorithms that solve it. Studying explicitly given algorithms is *analysis of algorithms*; studying problems is *computational complexity theory*. An algorithm's complexity is always an upper bound on the problem's complexity, and often the only thing known about a problem is that its complexity is no higher than the most efficient known algorithm — hence the large overlap between the two areas.
- **A function of input size.** Resource use grows with input size, so complexity is expressed as $n \mapsto f(n)$. Because complexity can vary dramatically between inputs of the same size, one distinguishes **worst-case** complexity (the maximum over all inputs of size $n$) and **average-case** complexity (the average over inputs of size $n$); unqualified "complexity" conventionally means worst-case time.
- **Machine-independent time.** Seconds are not used because they depend on the specific computer and on technological evolution. Instead one counts **elementary operations** ("steps"), each assumed to take constant time on a given machine and to change only by a constant factor across machines — capturing the *intrinsic* time requirements of the algorithm.
- **Asymptotics.** Exact worst- and average-case values are difficult to compute, change somewhat with the computer or model of computation, and matter little for small $n$ (where ease of implementation is often more interesting) — so complexity is expressed asymptotically for large $n$ using **big O notation**. Example: the usual integer-multiplication algorithm is $O(n^2)$, and the bound is sharp because the worst- and average-case complexities are also $\Omega(n^2)$; changing the number radix changes only the constants.
- **Other resources.** **Space complexity** measures memory required. For distributed algorithms executed by multiple interacting parties, the key resource is **communication complexity**. Counting arithmetic operations gives **arithmetic complexity**, which can differ sharply from **bit complexity**: computing the determinant of an $n \times n$ integer matrix by Gaussian elimination costs $O(n^3)$ arithmetic operations, but the bit complexity of those same algorithms is exponential in $n$ because intermediate coefficients may grow exponentially; coupling them with multi-modular arithmetic reduces bit complexity to $\tilde{O}(n^4)$. In sorting and searching, the resource generally counted is the number of entry comparisons.
- **Model of computation.** Evaluating complexity requires choosing which basic operations count as one unit-time step. When unspecified, a multitape Turing machine is implicitly assumed; more realistic models such as random-access machines are asymptotically equivalent for most problems, with explicit definitions needed only for very specific results (such as $O(n \log n)$ integer multiplication).

## Computational Models: The Theme of the Course

The heart of the course is **computational models**: using computation to help understand the world in which we live. Models act as *experimental devices* — they help us understand something that has happened, or predict the future. The slide's imagery makes the point vividly: where experiments were traditionally run in a physical laboratory, the arrow points to hands at a keyboard and mouse — the computer becomes our new laboratory, and experiments are run computationally rather than with beakers and instruments.

Three kinds of models will be covered this term:
1. **Optimization models** (today's topic),
2. **Statistical models**,
3. **Simulation models**.

Assigned reading: Section 12.1 of the text, plus Section 5.4 on lambda functions, which will be used shortly.

## Optimization Problems: Objectives, Constraints, and Search Spaces

**Definition.** Across mathematics, engineering, computer science, and economics, an optimization problem is the problem of finding the best solution from all feasible solutions.

**Anatomy (per the lecture).** An optimization model consists of two things:
1. An **objective function** to be maximized or minimized — e.g., minimize the time spent traveling from New York to Boston;
2. A set of **constraints** that must be honored — possibly empty, but usually not — e.g., spend no more than \$100, and be in Boston before 5:00 PM.

The travel sites on the slide (TripAdvisor, Priceline, Travelocity, Orbitz, Kayak, Hotwire, Expedia) are, in some sense, solving exactly this kind of problem for you: find a trip that satisfies your constraints and optimizes your objective.

**Search space.** The search space is the set of all possible points or solutions satisfying the problem's constraints, targets, or goals — the **feasible solutions** that can be evaluated against the objective function. It is often defined by the domain of the function being optimized, and it varies enormously in size and complexity: a continuous problem may have a multidimensional real-valued domain defined by bounds or constraints, while a discrete (combinatorial) problem's search space is a finite set of permutations, combinations, or configurations. In some contexts the term even refers to optimizing the domain itself — determining the most appropriate set of variables or parameters to define the problem. Understanding and navigating the search space is crucial for designing efficient algorithms, because it directly influences computational complexity and the likelihood of finding an optimal solution.

**Standard form (continuous).** By convention the standard form is a minimization,

$$
\begin{aligned}
\underset{x}{\operatorname{minimize}}\quad & f(x)\\
\operatorname{subject\ to}\quad & g_i(x)\le 0,\quad i=1,\dots,m\\
& h_j(x)=0,\quad j=1,\dots,p,
\end{aligned}
$$

where a maximization problem is treated by negating the objective function, and if $m=p=0$ the problem is unconstrained.

**Combinatorial form.** Formally, a combinatorial optimization problem $A$ is a quadruple $(I, f, m, g)$, and the goal for an instance $x$ is an optimal solution — a feasible $y$ achieving the best measure,

$$m(x,y)=g\left\{m(x,y') : y'\in f(x)\right\}.$$

**Optimization vs. decision versions.** Each combinatorial optimization problem has a corresponding **decision problem** asking whether a feasible solution exists for some particular measure threshold $m_0$. Example: "find a path from $u$ to $v$ that uses the fewest edges" might have the answer 4, while the corresponding decision problem asks "is there a path from $u$ to $v$ that uses 10 or fewer edges?" — answerable with a simple yes or no. In approximation algorithms, where the aim is near-optimal solutions to hard problems, the usual decision version is inadequate because it only specifies acceptable solutions; even when suitable decision problems can be introduced, the problem is more naturally characterized as an optimization problem.

## The Knapsack Problem: Statement and Variants

**The story.** One of the most famous classes of optimization problems. A burglar breaks into a house carrying a knapsack of limited capacity, surrounded by loot — a Picasso painting, a gold crown, a big stack of 500-euro bills, an Egyptian statue, a gold bar, an antique clock — and it will not all fit. What is the optimal choice of items to take?

**Why it matters generally.** The name derives from someone constrained by a fixed-size knapsack who must fill it with the most valuable items. The problem arises in resource allocation wherever decision-makers must choose from a set of **non-divisible** projects or tasks under a fixed budget or time constraint. It has been studied for more than a century, with early works dating back to 1897.

**0-1 knapsack.** The most common version restricts the number $x_i$ of copies of each kind of item to zero or one. Given $n$ items numbered $1$ to $n$, each with weight $w_i$ and value $v_i$, and a maximum weight capacity $W$:

$$
\underset{x}{\operatorname{maximize}}\ \sum_{i=1}^{n} v_i x_i
\qquad\text{subject to}\qquad
\sum_{i=1}^{n} w_i x_i \le W,\quad x_i\in\{0,1\}.
$$

Informally: maximize the sum of the values of the items in the knapsack so that the sum of the weights does not exceed the capacity.

**Variants.**
- **Bounded knapsack problem (BKP)**: removes the one-copy restriction but caps $x_i$ at a maximum non-negative integer $c$.
- **Unbounded knapsack problem (UKP)**: places no upper bound on the number of copies of each kind of item; $x_i$ need only be a non-negative integer.

**Subset sum.** The special case of the decision and 0-1 problems in which, for each item, weight equals value: $w_i = v_i$. In cryptography, "knapsack problem" often refers specifically to subset sum, which is one of Karp's 21 NP-complete problems.

## Why Knapsack Is Hard: The Complexity Landscape

**Decision ↔ optimization equivalence.** If a polynomial algorithm solves the decision version, the maximum value of the optimization version can be found in polynomial time by applying that algorithm iteratively while increasing $k$; conversely, a polynomial-time optimizer solves the decision problem in polynomial time by comparing its output value with $k$. Both versions are therefore of similar difficulty.

**Weak vs. strong NP-completeness.** Hardness depends on the form of the input: with integer weights and profits the problem is **weakly NP-complete**, while with rational weights and profits it is **strongly NP-complete** — yet the rational case still admits a fully polynomial-time approximation scheme (FPTAS).

**Dependence on the computational model.** The NP-hardness relates to models in which the size of integers matters (such as the Turing machine). Decision trees count each decision as a single step: Dobkin and Lipton showed a $\tfrac{1}{2}n^2$ lower bound on *linear* decision trees (decision nodes test the sign of affine functions) for knapsack; Steele and Yao generalized this to algebraic decision trees; and the lower bound extends to the real random-access machine with addition, subtraction, multiplication, comparison, and either division or floor — a model that covers more algorithms (including indexing into tables) and counts all program steps, not just decisions. On the upper-bound side, Meyer auf der Heide showed that for every $n$ there exists an $O(n^4)$-deep linear decision tree solving subset-sum with $n$ items — though this per-$n$ existence result implies no upper bound for an algorithm solving the problem for any given $n$.

**Hard instances.** A research theme is identifying what the "hard" instances look like — or, viewed another way, which properties of instances in practice make them more amenable than worst-case NP-complete behavior suggests. A key motivation is public-key cryptography: systems such as the Merkle–Hellman knapsack cryptosystem want hard instances. More generally, better understanding of the structure of an optimization problem's instance space advances the study of the problem and can improve algorithm selection.

## Solving Knapsack: Algorithms and Real-World Applications

**Algorithm families.** Available methods are based on dynamic programming, branch and bound, or hybridizations of the two.

**Dynamic programming for UKP.** Let $m[w]$ denote the best value achievable with capacity $w$. Two properties characterize it:
1. $m[0]=0$ — the sum of zero items, i.e., the summation of the empty set;
2. For remaining capacity $w$,
$$m[w]=\max\bigl(v_1+m[w-w_1],\ v_2+m[w-w_2],\ \dots,\ v_n+m[w-w_n]\bigr),$$
taken over item types with $w_i \le w$.

Why this works: during the run of the method, weight $w$ can be reached in only those ways — the previous weights are exactly $w-w_1, w-w_2, \dots, w-w_i$ for the $i$ item kinds. Whichever item type was placed last, say type $i$, leaves precisely the subproblem of capacity $w-w_i$, so maximizing $v_i + m[w-w_i]$ over all admissible types is exhaustive.

**Applications.**
- **Cutting stock**: finding the least wasteful way to cut raw materials.
- **Finance**: selection of investments and portfolios; selection of assets for asset-backed securitization.
- **Cryptography**: generating keys for the Merkle–Hellman and other knapsack cryptosystems.
- **Test construction and scoring** (an early application): if an exam has 12 questions worth 10 points each, letting test-takers answer any 10 for a maximum of 100 is fairly simple. With a heterogeneous distribution of point values it is harder: Feuerman and Weiss proposed a system with a test worth 125 total points, where students answer everything to the best of their abilities and a knapsack algorithm determines, among the subsets of problems whose point values add up to 100, the one giving each student the highest possible score.

**Standing in the field.** A 1999 study of the Stony Brook University Algorithm Repository found that, out of 75 algorithmic problems related to combinatorial algorithms and algorithm engineering, the knapsack problem was the 19th most popular and the third most needed, after suffix trees and the bin packing problem.

In [4]:
from pipeline.pdf import save_pdf

save_pdf(md, Path("results/lecture.pdf"))

WindowsPath('results/lecture.pdf')

Run the full course (MIT 6.0002, 15 lectures, 531 slides) under `data/`.

**This is a long, paid run.**


In [ ]:
import os
from runs import export_pdfs, unified_lectures, unify

from runs import (
    RunAborted,
    lecture_name,
    lectures,
    process_deck,
    results_dir,
    summarize,
)

MAX_SLIDES = None      # None for the whole deck
SKIP_DONE = True       # skip lectures that already have a stats.json

base = results_dir(MODEL)
for pdf in lectures():
    if SKIP_DONE and (base / lecture_name(pdf) / "stats.json").exists():
        print(f"skip {lecture_name(pdf)}")
        continue
    try:
        await process_deck(pdf, MODEL, max_slides=MAX_SLIDES, critic_model=CRITIC)
    except RunAborted as e:
        print(f"!! {e} - stopping.")
        break

summarize(MODEL)


for d in unified_lectures(MODEL):
    await unify(d, MODEL)

export_pdfs(MODEL, skip_done=True)